<a href="https://colab.research.google.com/github/ljcheng84/JC-NUS-ISS-ARCHAAS-STE-2026/blob/main/Automated_Tender_%26_Solution_Architecture_Proposal_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Automated Tender & Solution Architecture Proposal Engine

##Overview

This project implements a multi-agent system designed to automate the process of responding to IT infrastructure tenders.
By utilizing a sequential state-graph architecture, the system ingests user-uploaded tender specifications, designs a technical architecture topology, and outputs a commercial Bill of Materials (BOM).

##3 Distinct Agents Personas

Requirements Analyst: Extracts compliance and SLA constraints from uploaded documents.

Solution Architect: Designs hybrid/multi-cloud topologies.

Commercial Manager: Generates hardware/software BOM and pricing.

##Inter-Agent Coordination

Implemented using a sequential LangGraph StateGraph pipeline.

Uses conditional routing (add_conditional_edges) to handle tool execution loops before advancing the state from Analyst -> Architect -> Commercial.

##Tool Integration & Access Control

Each agent is bound to a specific, unique tool.

Implemented dedicated ToolNode instances for each agent (analyst_tools, architect_tools, commercial_tools) to physically separate execution environments and enforce strict access boundaries.

extract_tender_specifications (Analyst only): Parses text from uploaded PDFs/documents.

search_architecture_patterns (Architect only): Mocks a database lookup for compliant enterprise architecture blueprints.

calculate_bom_pricing (Commercial only): Calculates infrastructure costs based on the generated design.

##State Management

Utilizes a shared Python TypedDict (TenderState) to pass conversation history, document content, and structured extracted data (requirements, architecture, BOM) between agents.

Each agent reads directly from the specific output field of its predecessor.

##System Architecture

The system coordinates agents using LangGraph in Python.

State: A shared dictionary tracking messages, uploaded_document_text, compliance_requirements, architecture_design, and final_bom.

##Models Used:

gpt-5-mini (Temperature: 1.0) - Used by Requirements Analyst.

gpt-4o-mini (Temperature: 0.2) - Used by Solution Architect for deterministic technical mapping.

gpt-5-nano (Temperature: 1.0) - Used by Commercial Manager.

#Setup Instructions (Google Colab)

This project is designed to run seamlessly in a Google Colab notebook.

##1. Install Dependencies

Run the following in your first Colab cell to install the framework and document processing libraries:

%pip install -q langchain-openai langgraph pypdf


##2. Configure Environment Variables

You must set your OpenAI API key. In Google Colab, it is recommended to use the built-in Secrets manager.

import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


##3. Upload Tender Specifications

Use Google Colab's file upload widget to provide the raw tender documents (e.g., PDF) to the local filesystem before running the agent graph.

from google.colab import files
uploaded = files.upload()


##4. Run the Workflow

Execute the cells defining the tools, the TenderState, the agent nodes, conditional routers, and the graph compilation.

Invoke the graph, providing the filename of the uploaded specification document and initializing the required state fields:

initial_state = {
    "messages": [("user", "Draft a proposal based on the uploaded file: 'tender_spec.pdf'.")],
    "compliance_requirements": "",
    "architecture_design": "",
    "final_bom": ""
}
result = app.invoke(initial_state)

print(result["final_bom"])


#Repository Structure

proposal_engine.ipynb: The main Google Colab notebook containing all agent definitions, document upload logic, conditional routers, tool mocks, and graph routing.

README.md: Project documentation.$

# Step 1 - Installations & Environment Setup

In [1]:
%pip install -q langchain-openai langgraph pypdf

import os
from google.colab import userdata

# Load OpenAI API Key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Step 2a - Define the Mock Data for each Personas

In [2]:
# @title
from langchain_core.tools import tool

@tool
def extract_tender_specifications(filename: str) -> str:
    """Parses text from uploaded tender PDFs/documents."""
    # Mocking document extraction for the Analyst
    return """Extracted Specs:
    - 500-user Virtual Desktop Infrastructure (VDI).
    - Requires zero-trust network access.
    - SLA: 99.99% uptime with an on-premises disaster recovery site.
    - Strict endpoint detection and response (EDR) integration."""

@tool
def search_architecture_patterns(requirements: str) -> str:
    """Looks up compliant enterprise architecture blueprints based on requirements."""
    # Mocking the architectural database lookup for the Architect
    return """Blueprint Match:
    - Virtualization: Nutanix AHV for on-prem VDI.
    - Identity & Access: Omnissa Workspace ONE integrated with Microsoft Entra ID.
    - Security: Check Point Quantum firewalls for zero-trust zoning and Trellix EDR."""

@tool
def calculate_bom_pricing(architecture: str) -> str:
    """Calculates infrastructure hardware/software costs and rack power ratings."""
    # Mocking the commercial calculations for the Commercial Manager
    return """Pricing Model Generated:
    - Hardware BOM: 3x Nutanix HCI Nodes ($120,000 CapEx).
    - Rack Profile: 2U space, 1.5kW per node.
    - Licensing: Omnissa & Check Point subscriptions ($3,500/month).
    - Leasing Option: Device-as-a-Service model at $6,200/month over 36 months."""

tools_list = [extract_tender_specifications, search_architecture_patterns, calculate_bom_pricing]

# Step 2b - Each Persona use LLM for actual inference

In [ ]:
import pypdf
from langchain_core.tools import tool

@tool
def extract_tender_specifications(filename: str) -> str:
    """Reads and extracts raw text from an uploaded PDF tender document."""
    try:
        text = ""
        with open(filename, "rb") as f:
            reader = pypdf.PdfReader(f)
            for page in reader.pages:
                text += page.extract_text() + "\n"
        return text
    except Exception as e:
        return f"Error reading file {filename}: {str(e)}. Please ensure the file is uploaded correctly."

@tool
def search_architecture_patterns(query: str) -> str:
    """Provides enterprise architecture guidelines and best practices based on a search query."""
    # Provides raw reference data so the Architect LLM has to design the actual topology
    knowledge_base = {
        "vdi": "Virtual Desktop Infrastructure best practices: Deploy on Nutanix AHV for on-premises, or Azure Virtual Desktop for cloud. Require Omnissa Horizon for VDI management.",
        "zero-trust": "Zero-trust architecture mandates micro-segmentation (e.g., Check Point Quantum firewalls) and endpoint detection and response (e.g., Trellix EDR).",
        "hpc": "High-Performance Computing requires NVIDIA GPU nodes, RoCE lossless networking, and Slurm workload managers.",
        "hybrid": "Hybrid identity synchronization should utilize Microsoft Entra Cloud Sync with Active Directory."
    }

    results = [val for key, val in knowledge_base.items() if key in query.lower()]
    if results:
        return "Reference Guidelines:\n" + "\n".join(results)
    return "No specific architectural guidelines found for this query. Rely on standard enterprise cloud best practices."

@tool
def calculate_bom_pricing(components: list[str]) -> str:
    """Looks up unit costs and physical power specifications for infrastructure components."""
    # Provides raw catalog prices so the Commercial LLM has to calculate the total BOM
    catalog = {
        "nutanix": {"cost": "$40,000 per node", "power": "1.5kW", "ru": "2U"},
        "azure": {"cost": "$500 per month per VM", "power": "N/A (Cloud)", "ru": "N/A"},
        "firewall": {"cost": "$15,000 per appliance", "power": "0.5kW", "ru": "1U"},
        "edr": {"cost": "$50 per user/year", "power": "N/A (Software)", "ru": "N/A"},
        "gpu": {"cost": "$30,000 per GPU", "power": "1.0kW", "ru": "Included in Node"}
    }

    results = []
    for comp in components:
        found = False
        for key, details in catalog.items():
            if key in comp.lower():
                results.append(f"Component matched '{key}': Unit Cost {details['cost']}, Power {details['power']}, Rack Space {details['ru']}")
                found = True
                break
        if not found:
            results.append(f"Component '{comp}': Pricing not found in standard catalog. Please estimate using market rates.")

    return "\n".join(results)

tools_list = [extract_tender_specifications, search_architecture_patterns, calculate_bom_pricing]

# Step 3 - State Definition & Personas

In [3]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import SystemMessage, BaseMessage
from langgraph.graph.message import add_messages

# Shared State across all agents
class TenderState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    compliance_requirements: str
    architecture_design: str
    final_bom: str

# Agent Personas
analyst_persona = SystemMessage(content="""You are a Lead IT Requirements Analyst.
Your sole responsibility is to analyze raw tender specifications and extract a strict compliance matrix.
Focus heavily on identifying zero-trust compliance standards, SLAs, RTO/RPOs, and network constraints.
You MUST use the extract_tender_specifications tool to read the file.
Do NOT propose a technical solution. Output a structured list of constraints.""")

architect_persona = SystemMessage(content="""You are a Principal Solution Architect.
Map the Analyst's compliance requirements into a concrete technical design.
You MUST use the search_architecture_patterns tool to find valid blueprints.
Explicitly define workload placements, virtualization platforms, and security flows.
Do NOT calculate costs. Output a comprehensive technical blueprint.""")

commercial_persona = SystemMessage(content="""You are a Commercial Presales Manager.
Translate the technical blueprint into a highly structured Bill of Materials (BOM) and commercial proposal.
You MUST use the calculate_bom_pricing tool.
Include physical data center specifications (rack power ratings) and commercial leasing structures.
Format as high-level executive presentation points.""")

# Step 4 - Agent Nodes and Graph compilation

In [4]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import ToolNode

# Initialize LLMs with the requested models and strict tool bindings
analyst_llm = ChatOpenAI(model="gpt-5-mini", temperature=1.0).bind_tools([extract_tender_specifications])
architect_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2).bind_tools([search_architecture_patterns])
commercial_llm = ChatOpenAI(model="gpt-5-nano", temperature=1.0).bind_tools([calculate_bom_pricing])

# --- 1. Define Agent Nodes ---
def analyst_node(state: TenderState):
    messages = [analyst_persona] + state["messages"]
    response = analyst_llm.invoke(messages)
    return {"messages": [response], "compliance_requirements": response.content}

def architect_node(state: TenderState):
    analyst_output = state.get("compliance_requirements", "")
    context = f"{architect_persona.content}\n\nAnalyst's Compliance Requirements:\n{analyst_output}"

    messages = [SystemMessage(content=context)] + state["messages"]
    response = architect_llm.invoke(messages)
    return {"messages": [response], "architecture_design": response.content}

def commercial_node(state: TenderState):
    architect_output = state.get("architecture_design", "")
    context = f"{commercial_persona.content}\n\nArchitect's Technical Blueprint:\n{architect_output}"

    messages = [SystemMessage(content=context)] + state["messages"]
    response = commercial_llm.invoke(messages)
    return {"messages": [response], "final_bom": response.content}

# --- 2. Define Conditional Routers ---
# These functions check if the agent called a tool or finished its task
def route_analyst(state: TenderState):
    if state["messages"][-1].tool_calls:
        return "analyst_tools"
    return "architect"

def route_architect(state: TenderState):
    if state["messages"][-1].tool_calls:
        return "architect_tools"
    return "commercial"

def route_commercial(state: TenderState):
    if state["messages"][-1].tool_calls:
        return "commercial_tools"
    return END

# --- 3. Build the Graph ---
workflow = StateGraph(TenderState)

# Add Agent Nodes
workflow.add_node("analyst", analyst_node)
workflow.add_node("architect", architect_node)
workflow.add_node("commercial", commercial_node)

# Add Dedicated Tool Nodes (Enforces strict access control)
workflow.add_node("analyst_tools", ToolNode([extract_tender_specifications]))
workflow.add_node("architect_tools", ToolNode([search_architecture_patterns]))
workflow.add_node("commercial_tools", ToolNode([calculate_bom_pricing]))

# --- 4. Set Edges and Routing ---
workflow.set_entry_point("analyst")

# Analyst Execution Loop
workflow.add_conditional_edges("analyst", route_analyst)
workflow.add_edge("analyst_tools", "analyst") # Return to analyst after tool execution

# Architect Execution Loop
workflow.add_conditional_edges("architect", route_architect)
workflow.add_edge("architect_tools", "architect")

# Commercial Execution Loop
workflow.add_conditional_edges("commercial", route_commercial)
workflow.add_edge("commercial_tools", "commercial")

app = workflow.compile()

# Step 5 - Upload Tender Documentation

In [ ]:
# Initialize the state with a human request pointing to a dummy uploaded file
initial_state = {
    "messages": [("user", "Please process the uploaded tender specification document: 'MoD_VDI_Tender_2026.pdf'")],
    "compliance_requirements": "",
    "architecture_design": "",
    "final_bom": ""
}

# Run the multi-agent graph
result = app.invoke(initial_state)


# Step 6a - Show the output from Persona - ANALYST


In [24]:
print("=== STEP 1: Tender Requirements Analysis Output ===")
print(result["messages"][3].content)


=== STEP 1: Tender Requirements Analysis Output ===
I have read the uploaded tender extract and produced a strict, measurable compliance matrix focused on zero‑trust, SLAs, RTO/RPOs and network constraints. I did NOT propose any technical solution — only extracted and formalised requirements and identified missing/unresolved specification elements that MUST be clarified.

Summary of source-extracted items used
- 500‑user Virtual Desktop Infrastructure (VDI).
- Requires zero‑trust network access.
- SLA: 99.99% uptime with an on‑premises disaster recovery site.
- Strict endpoint detection and response (EDR) integration.

Compliance matrix (strict, testable constraints)

1) ID: R-001
- Requirement: Support for 500 concurrent VDI user sessions.
- Category: Capacity / Functional
- Priority: MUST
- Acceptance criteria: System must demonstrate sustained operation supporting 500 concurrent authenticated user sessions for a continuous 8‑hour business period without session drops caused by capac

# Step 6b - Show the output from Persona - SOLUTION ARCHITECT

In [26]:

print("=== Step 2: Proposed Technical Solution  ===")
print(result["messages"][6].content)


=== Step 2: Proposed Technical Solution  ===
Based on the compliance requirements and the architecture patterns identified, here is a comprehensive technical blueprint for the proposed solution:

### Technical Blueprint for VDI Solution

#### 1. **Workload Placements**
- **Primary VDI Environment**: 
  - **Platform**: Nutanix AHV
  - **Capacity**: Designed to support 500 concurrent user sessions.
  - **Location**: On-premises data center.

- **Disaster Recovery (DR) Environment**:
  - **Platform**: Nutanix AHV (mirroring primary environment)
  - **Location**: Physically separate on-premises DR site.

#### 2. **Virtualization Platforms**
- **Primary and DR Sites**: 
  - **Hypervisor**: Nutanix AHV
  - **Management**: Nutanix Prism for centralized management of VDI resources.

#### 3. **Identity and Access Management**
- **Centralized Authentication**:
  - **Solution**: Omnissa Workspace ONE integrated with Microsoft Entra ID.
  - **MFA**: Multi-factor authentication enforced for all VDI

# Step 6c - Show the output from Persona - COMMERCIAL MANAGER

In [25]:

print("=== Step 3: Proposed COMMERCIAL BOM ===")
print(result["messages"][9].content)

=== Step 3: Proposed COMMERCIAL BOM ===
Executive BOM and Commercial Proposal – MoD VDI Tender 2026

Purpose
- Translate tender requirements into a structured, vendor-ready BOM and commercial model.
- Align 500-user on-prem VDI with zero-trust, DR, EDR, and SOC telemetry requirements while presenting clear CapEx/OpEx options and data-center implications.

Summary alignment to tender
- On-prem VDI cluster (Nutanix AHV) designed for 500 concurrent sessions.
- DR site at a physically separate on-prem location with mirroring capability.
- Centralized IAM with MFA (Workspace ONE + Microsoft Entra ID) under Zero-Trust.
- ZTNA with Check Point Quantum firewalls; segmentation between user and management traffic.
- EDR integration (Trellix) across infrastructure and VDI endpoints.
- Encryption in transit; detailed network segmentation; SOC telemetry and 99.99% uptime SLA target.
- Data residency on-premises; DR failover testing for 500 users; acceptance testing coverage.

Bill of Materials (hig